# Reference KB — Query → Encode → Retrieve → Rerank → Display

A query-time companion to `test_reference_ingest.ipynb`. It does **not** ingest —
it runs the **retrieval + reranking** pipeline over the reference book index that
`ingest_references.py` already built (`data/reference_chroma`, collection
`threat_modeling_refs`).

```
 user query
     │  (1) encode  ── local embedding model (bge-small, GPU auto)
     ▼
 query vector ──(2) retrieve top-k ── ChromaDB cosine search ──▶ candidate chunks
                                                                      │
                                              (3) rerank ── cross-encoder scores
                                                  (query, passage) pairs          │
                                                                      ▼
                                              (4) display ── final ranked passages
```

**Why rerank?** Dense retrieval uses a *bi-encoder* (query and passages embedded
**separately**, compared by cosine) — fast over the whole corpus but approximate.
A **cross-encoder** reranker reads the **query and a candidate together**, so it
scores relevance far more precisely. The standard recipe: retrieve a *wide* top-k
cheaply, then rerank to a *narrow* final-k accurately.

## Prerequisites
1. Ingest the books first (from the project dir):
   ```bash
   python ingest_references.py        # builds data/reference_chroma
   ```
2. Deps: `chromadb`, `sentence-transformers`, `torch`, plus this notebook's
   reranker (also `sentence-transformers`) and `pandas` for the tables.
3. A GPU is optional (auto-detected); CPU works, just slower. The reranker model
   downloads once (~80 MB for the default MiniLM cross-encoder).

> Run this with the **same interpreter/kernel** that wrote `data/reference_chroma`
> (matching ChromaDB version), or Chroma may fail to open the store.

In [1]:
import logging
import sys
from pathlib import Path

# Make the threat_model package importable when running from the project dir.
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from dotenv import load_dotenv
load_dotenv()
logging.basicConfig(level="INFO", format="%(levelname)-7s | %(message)s")
# Quiet the chatty HTTP/model-download loggers so the notebook output stays readable.
for noisy in ("httpx", "httpcore", "urllib3", "huggingface_hub", "filelock",
              "sentence_transformers", "chromadb.telemetry"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

import torch
import pandas as pd
import threat_model.references as R

DEVICE = R._device()           # cuda > mps > cpu (same policy the ingest uses)
print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("embedding model:", R.EMBEDDING_MODEL)

# Render full text in pandas cells (don't truncate mid-word).
pd.set_option("display.max_colwidth", 110)

torch: 2.11.0+cu128 | device: cuda
GPU: NVIDIA GeForce RTX 5090
embedding model: BAAI/bge-small-en-v1.5


## 1. Connect to the reference index
We open the **existing** ChromaDB collection (read-only use here — no ingest).

In [2]:
kb = R.ReferenceKB()
n = kb.count()
print(f"Reference collection '{kb.collection_name}' holds {n} chunks.")
assert n > 0, (
    "The reference collection is empty. Run `python ingest_references.py` first "
    "(see Prerequisites)."
)

Reference collection 'threat_modeling_refs' holds 2801 chunks.


## 2. Your query
Edit `QUERY` below. (For an interactive prompt, flip `INTERACTIVE = True` — kept
off by default so the notebook also runs headless via `nbconvert --execute`.)

In [3]:
INTERACTIVE = False
QUERY = "How should I threat model a system that processes payments?"

if INTERACTIVE:
    try:
        QUERY = input("Enter your query: ").strip() or QUERY
    except Exception:
        pass
print("Query:", QUERY)

Query: How should I threat model a system that processes payments?


## 3. Encode the query (embedding)
We turn the text into a vector with the **same local model** used at ingest time,
so query and passages live in the same space. `kb.search()` does this internally;
here we call the encoder explicitly to inspect the vector.

In [4]:
qvec = R._embed_query(QUERY)          # uses the bge query instruction + normalize
import numpy as np
qvec_np = np.asarray(qvec, dtype="float32")
print("vector length :", qvec_np.shape[0])
print("L2 norm       :", round(float(np.linalg.norm(qvec_np)), 4), "(≈1.0 — normalized)")
print("first 8 dims  :", np.round(qvec_np[:8], 4).tolist())

/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO    | Loading reference embedding model 'BAAI/bge-small-en-v1.5' on device 'cuda'…


WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14640.70it/s]

INFO    | GPU: NVIDIA GeForce RTX 5090


vector length : 384
L2 norm       : 1.0 (≈1.0 — normalized)
first 8 dims  : [-0.008700000122189522, -0.019600000232458115, -0.06270000338554382, 0.003700000001117587, 0.04960000142455101, -0.07450000196695328, 0.04839999973773956, 0.014800000004470348]


## 4. Retrieve top-k candidates (dense vector search)
Pull a **wide** set of candidates (`RETRIEVE_K`) by cosine similarity. These are
ranked by the bi-encoder score — good recall, approximate ordering.

In [5]:
RETRIEVE_K = 20

candidates = kb.search(QUERY, k=RETRIEVE_K)   # [{text, metadata, score}], score=cosine sim
print(f"Retrieved {len(candidates)} candidates.\n")

def _snippet(t, n=110):
    return " ".join(t.split())[:n]

retrieved_df = pd.DataFrame([
    {
        "retr_rank": i,
        "dense_score": round(h["score"], 4),
        "source": h["metadata"].get("source", "?"),
        "page": h["metadata"].get("page", "?"),
        "snippet": _snippet(h["text"]),
    }
    for i, h in enumerate(candidates, start=1)
])
retrieved_df

Retrieved 20 candidates.



,retr_rank,dense_score,source,page,snippet
0,1,0.7626,threat_modeling_designing_for_security,564,"(That is, use the software model in Figure E-1 and the operational model in Figure E-2 and fi nd threats a..."
1,2,0.7574,threat_modeling_designing_for_security,216,"when you don’t have a response planned. For example, if you operate banking software, you probably have an..."
2,3,0.7556,threat_modeling_designing_for_security,164,possible) threat modeling. Begin the process with the component(s) on which the participants in the room a...
3,4,0.7352,threat_modeling_designing_for_security,66,"same systems. Applying the question “what’s your threat model?” to the Acme Corporation example, you might..."
4,5,0.7351,threat_modeling_designing_for_security,415,"realize that you’re listening to them, which in turn helps them “buy in” and support your plan. Obviously,..."
5,6,0.7316,threat_modeling_designing_for_security,163,"project is a long one.) Close to Delivery Lastly, you should threat model as you get ready to ship by reex..."
6,7,0.7270,threat_modeling_designing_for_security,49,"a police state, but I don’t advocate their use. ■ Transferring threats is about letting someone or somethi..."
7,8,0.7240,threat_modeling_designing_for_security,255,"That is, taken to an extreme, it quickly becomes ridiculous. For example, if a product were to declare tha..."
8,9,0.7238,threat_modeling_designing_for_security,66,"of standard answers to “what’s your threat model?” in Appendix A, “Helpful Tools,” but a few examples are ..."
9,10,0.7232,AutomotiveThreatModeling,157,"the system may not be limited to interfaces. Internal components can also be the source of a threat, such ..."


## 5. Rerank with a cross-encoder
Score every `(query, passage)` pair jointly with a cross-encoder, then re-sort.
Default is the lightweight `cross-encoder/ms-marco-MiniLM-L-6-v2`; for higher
quality (and a bigger download) swap in `BAAI/bge-reranker-base` or
`BAAI/bge-reranker-v2-m3`.

In [6]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
FINAL_K = 5

reranker = CrossEncoder(RERANKER_MODEL, device=DEVICE)   # downloads once, cached
pairs = [(QUERY, h["text"]) for h in candidates]
rerank_scores = reranker.predict(pairs, show_progress_bar=False)

# Attach scores + original retrieval rank, then sort by the cross-encoder score.
scored = [
    {**h, "dense_score": h["score"], "rerank_score": float(s), "retr_rank": i}
    for i, (h, s) in enumerate(zip(candidates, rerank_scores), start=1)
]
reranked = sorted(scored, key=lambda x: x["rerank_score"], reverse=True)

rerank_df = pd.DataFrame([
    {
        "rerank_rank": j,
        "retr_rank": h["retr_rank"],
        "moved": h["retr_rank"] - j,          # +ve = promoted by reranking
        "rerank_score": round(h["rerank_score"], 4),
        "dense_score": round(h["dense_score"], 4),
        "source": h["metadata"].get("source", "?"),
        "page": h["metadata"].get("page", "?"),
        "snippet": _snippet(h["text"]),
    }
    for j, h in enumerate(reranked[:FINAL_K], start=1)
])
print(f"Top {FINAL_K} after reranking (of {len(candidates)} candidates):")
rerank_df

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13144.76it/s]

Top 5 after reranking (of 20 candidates):


,rerank_rank,retr_rank,moved,rerank_score,dense_score,source,page,snippet
0,1,4,3,3.5794,0.7352,threat_modeling_designing_for_security,66,"same systems. Applying the question “what’s your threat model?” to the Acme Corporation example, you might..."
1,2,18,16,1.4010,0.7153,threat_modeling_designing_for_security,179,Chapter 7 ■ Processing and Managing Threats 143 c07.indd 09:44:3:AM 01/09/2014 Page 143 Summary There are ...
2,3,13,10,1.3938,0.7199,threat_modeling_designing_for_security,28,learn about how to bring threat modeling into your development process. And a whole lot more! Systems Arch...
3,4,16,12,1.3593,0.7170,threat_modeling_designing_for_security,165,Chapter 7 ■ Processing and Managing Threats 129 c07.indd 09:44:3:AM 01/09/2014 Page 129 then work across t...
4,5,6,1,1.3079,0.7316,threat_modeling_designing_for_security,163,"project is a long one.) Close to Delivery Lastly, you should threat model as you get ready to ship by reex..."


## 6. Display the final reranked passages
The narrow, high-precision result set — with citations — ready to hand to an LLM
as grounding context or to read directly.

In [7]:
def show_results(results, query, k=FINAL_K, width=600):
    print(f"QUERY: {query}\n" + "=" * 80)
    for j, h in enumerate(results[:k], start=1):
        m = h["metadata"]
        cite = f'{m.get("source", "book")}, p.{m.get("page", "?")}'
        text = " ".join(h["text"].split())
        if len(text) > width:
            text = text[:width] + " …"
        print(f"\n[{j}] {cite}   rerank={h['rerank_score']:.4f}  (dense={h['dense_score']:.4f})")
        print("    " + text)

show_results(reranked, QUERY)

QUERY: How should I threat model a system that processes payments?

[1] threat_modeling_designing_for_security, p.66   rerank=3.5794  (dense=0.7352)
    same systems. Applying the question “what’s your threat model?” to the Acme Corporation example, you might get the following answers: ■ For the Acme SQL database, the threat model would be an attacker who wants to read or change data in the database. A more subtle model might also include people who want to read the data without showing up in the logs. ■ For Acme’s fi nancial system, the answers might include someone getting a check they didn’t deserve, customers who don’t make a payment they owe, and/or someone reading or altering fi nancial results before reporting. If you don’t have a clear …

[2] threat_modeling_designing_for_security, p.179   rerank=1.4010  (dense=0.7153)
    Chapter 7 ■ Processing and Managing Threats 143 c07.indd 09:44:3:AM 01/09/2014 Page 143 Summary There are a set of tools and techniques that you can use to h

## 7. Reusable helper
The whole pipeline as one function so you can try other queries quickly.

In [8]:
_RERANKER_CACHE = {}

def query_rerank(query, retrieve_k=20, final_k=5, reranker_model=RERANKER_MODEL):
    """Encode → retrieve top-k → cross-encoder rerank → return final_k passages."""
    cands = kb.search(query, k=retrieve_k)
    if not cands:
        return []
    if reranker_model not in _RERANKER_CACHE:
        _RERANKER_CACHE[reranker_model] = CrossEncoder(reranker_model, device=DEVICE)
    rr = _RERANKER_CACHE[reranker_model]
    scores = rr.predict([(query, c["text"]) for c in cands], show_progress_bar=False)
    out = [
        {**c, "dense_score": c["score"], "rerank_score": float(s)}
        for c, s in zip(cands, scores)
    ]
    out.sort(key=lambda x: x["rerank_score"], reverse=True)
    return out[:final_k]

demo_q = "What are common spoofing threats and how do I mitigate them?"
show_results(query_rerank(demo_q), demo_q)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14182.27it/s]

QUERY: What are common spoofing threats and how do I mitigate them?

[1] threat_modeling_designing_for_security, p.48   rerank=4.9986  (dense=0.8276)
    to address these threats, see Chapters 8 and 9, “Defensive Building Blocks” and “Tradeoffs When Addressing Threats.” ■ Mitigating threats is about doing things to make it harder to take advan- tage of a threat. Requiring passwords to control who can log in mitigates the threat of spoofi ng. Adding password controls that enforce complex- ity or expiration makes it less likel y that a password will be guessed or usable if stolen. ■ Eliminating threats is almost always achieved by eliminating features. If you have a threat that someone will access the administrative function of

[2] threat_modeling_designing_for_security, p.201   rerank=3.4116  (dense=0.7789)
    Chapter 8 ■ Defensive Tactics and Technologies 165 c08.indd 12:26:46:PM 01/13/2014 Page 165 technologies are available to address each of the STRIDE threats. There are tactics a

## Notes
- **Bi-encoder (retrieval) vs cross-encoder (reranking):** the bi-encoder embeds
  query and passages *independently* so the whole corpus is searchable in one ANN
  lookup; the cross-encoder reads *query + passage together* for a sharper score
  but is too slow to run over the full corpus — hence retrieve-wide-then-rerank.
- **Tuning:** raise `RETRIEVE_K` for better recall before reranking; lower
  `FINAL_K` for a tighter context window. Heavier rerankers
  (`BAAI/bge-reranker-*`) trade speed for quality.
- **Grounding:** this is the same retrieval step the app uses in OpenAI mode
  (`references.retrieve_reference_context`), with a reranking stage added — drop
  the reranked passages into `build_threat_model_prompt(..., reference_context=...)`.
- Run with the interpreter/kernel that wrote `data/reference_chroma` (matching
  ChromaDB version), or the store may not open.

In [9]:
# Environment sanity check
print("device          :", DEVICE)
print("collection chunks:", kb.count())
print("retrieved        :", len(candidates))
print("reranked (final) :", min(FINAL_K, len(candidates)))
print("OK — query → encode → retrieve → rerank → display ran end-to-end.")

device          : cuda
collection chunks: 2801
retrieved        : 20
reranked (final) : 5
OK — query → encode → retrieve → rerank → display ran end-to-end.
